# ASG Airlines – End-to-End Data Engineering Case Study

## Objective

This project builds an end-to-end data engineering pipeline for ASG Airlines.

we give the overall pipeline performs of this case study ,and i list it here so following steps are used:

1. Data ingestion
2. Data profiling
3. Data quality assessment
4. Data cleaning
5. Data standardization
6. Flight duration calculation
7. Overnight flight handling
8. PII protection
9. Data validation
10. Analytical dataset creation
11. KPI preparation
12. Export for Power BI

In [52]:
#Install/import libraries
import pandas as pd
import numpy as np
import re
import hashlib
import os
import logging
from datetime import timedelta

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
from google.colab import files

uploaded = files.upload()

Saving UseCase - Airlines.xlsx to UseCase - Airlines.xlsx


In [53]:
#Load the four datasets
file_name = "UseCase - Airlines.xlsx"

flights = pd.read_excel(file_name, sheet_name="flights")
payments = pd.read_excel(file_name, sheet_name="payments")
bookings = pd.read_excel(file_name, sheet_name="bookings")
passengers = pd.read_excel(file_name, sheet_name="passengers")

print("Flights:", flights.shape)
print("Bookings:", bookings.shape)
print("Passengers:", passengers.shape)
print("Payments:", payments.shape)

Flights: (1020, 7)
Bookings: (1000, 9)
Passengers: (1039, 9)
Payments: (1000, 4)


In [54]:
#Display the datasets
print("FLIGHTS")
display(flights.head())

print("BOOKINGS")
display(bookings.head())

print("PASSENGERS")
display(passengers.head())

print("PAYMENTS")
display(payments.head())

FLIGHTS


,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00


BOOKINGS


,booking_id,passenger_id,flight_id,booking_date,status,passport_number,seat_number,emergency_contact_name,emergency_contact_phone
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,P1945887,3D,Isaac Bakshi,+91-6478475128
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,L3482012,18A,Anvi Konda,+91-6647078662
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,G8507659,30C,Udant Dewan,+91-8405938220
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,M0891776,33A,Harsh Chahal,+91-6264636839
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,N5742231,25C,Pahal Balay,+91-9336478266


PASSENGERS


,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
0,P1000,Vivaan,Chatterjee,52,F,vivaan.chatterjee@gmail.com,+91-6896233790,433218196001,1974-04-08
1,P1001,Krishna,Reddy,15,M,krishna.reddy@hotmail.com,+91-6702632297,386379402654,2011-03-07
2,P1002,Myra,Naidu,72,M,myra.naidu@outlook.com,+91-6199585092,615594078161,1954-09-10
3,P1003,Myra,Mishra,61,F,myra.mishra@hotmail.com,+91-8719927151,310341316475,1965-03-12
4,P1004,Saanvi,Banerjee,21,M,saanvi.banerjee@outlook.com,+91-7819595113,419283276483,2005-11-11


PAYMENTS


,payment_id,booking_id,amount,payment_method
0,PAY1000,B1116,9883.49,NETBANKING
1,PAY1001,B1738,8457.96,NETBANKING
2,PAY1002,B1873,6495.37,UPI
3,PAY1003,B1914,5079.38,NETBANKING
4,PAY1004,B1967,12518.31,CARD


In [55]:
#Column information
datasets = {
    "Flights": flights,
    "Bookings": bookings,
    "Passengers": passengers,
    "Payments": payments
}

for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nData Types:")
    print(df.dtypes)


Flights
Rows: 1020
Columns: 7

Columns:
['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration']

Data Types:
flight_id                 object
airline                   object
source                    object
destination               object
departure_time    datetime64[ns]
arrival_time      datetime64[ns]
duration                  object
dtype: object

Bookings
Rows: 1000
Columns: 9

Columns:
['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'passport_number', 'seat_number', 'emergency_contact_name', 'emergency_contact_phone']

Data Types:
booking_id                         object
passenger_id                       object
flight_id                          object
booking_date               datetime64[ns]
status                             object
passport_number                    object
seat_number                        object
emergency_contact_name             object
emergency_contact_phone            object
dtype: object



## Data Quality Assessment

 check

- Missing values
- Duplicate records
- Invalid identifiers
- Invalid timestamps
- Inconsistent categorical values
- Referential integrity issues
- PII exposure

In [56]:
#Missing-value analysis
for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) == 0:
        print("No missing values found")
    else:
        print(missing)


Flights
airline    41
dtype: int64

Bookings
status    45
dtype: int64

Passengers
last_name    10
dtype: int64

Payments
amount    48
dtype: int64


In [57]:
#Missing-value summary table
missing_summary = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isnull().sum()
        missing_percentage = (missing_count / len(df)) * 100

        missing_summary.append({
            "Dataset": name,
            "Column": column,
            "Missing_Count": missing_count,
            "Missing_Percentage": round(missing_percentage, 2)
        })

missing_df = pd.DataFrame(missing_summary)

display(
    missing_df[missing_df["Missing_Count"] > 0]
    .sort_values("Missing_Count", ascending=False)
)

,Dataset,Column,Missing_Count,Missing_Percentage
27,Payments,amount,48,4.80
11,Bookings,status,45,4.50
1,Flights,airline,41,4.02
18,Passengers,last_name,10,0.96


In [58]:
#Duplicate analysis
for name, df in datasets.items():
    print(
        name,
        "duplicate rows:",
        df.duplicated().sum()
    )

Flights duplicate rows: 15
Bookings duplicate rows: 0
Passengers duplicate rows: 0
Payments duplicate rows: 0


In [59]:
print("Duplicate Flight IDs:", flights["flight_id"].duplicated().sum())
print("Duplicate Booking IDs:", bookings["booking_id"].duplicated().sum())
print("Duplicate Passenger IDs:", passengers["passenger_id"].duplicated().sum())
print("Duplicate Payment IDs:", payments["payment_id"].duplicated().sum())

Duplicate Flight IDs: 16
Duplicate Booking IDs: 0
Duplicate Passenger IDs: 39
Duplicate Payment IDs: 0


In [60]:
#Check categorical values
print("Airlines:")
print(flights["airline"].value_counts(dropna=False))

print("\nSources:")
print(flights["source"].value_counts(dropna=False))

print("\nDestinations:")
print(flights["destination"].value_counts(dropna=False))

print("\nBooking Status:")
print(bookings["status"].value_counts(dropna=False))

print("\nPayment Methods:")
print(payments["payment_method"].value_counts(dropna=False))

Airlines:
airline
IndiGo       249
SpiceJet     240
Air India    236
Vistara      223
NaN           41
UNKNOWN       31
Name: count, dtype: int64

Sources:
source
BOM    207
HYD    180
CCU    174
DEL    163
MAA    157
BLR    139
Name: count, dtype: int64

Destinations:
destination
DEL    200
CCU    188
BOM    174
BLR    165
MAA    151
HYD    142
Name: count, dtype: int64

Booking Status:
status
CONFIRMED    320
CANCELLED    314
PENDING      291
NaN           45
INVALID       30
Name: count, dtype: int64

Payment Methods:
payment_method
UPI           358
CARD          329
NETBANKING    313
Name: count, dtype: int64


## Flight ID Validation

Flight identifiers are checked for malformed or inconsistent values.

The validation rule is based on the observed airline flight-code structure.
Records that do not satisfy the validation pattern are flagged rather than
being assigned an artificial replacement value.

In [61]:
#Flight ID analysis
flights["flight_id"] = flights["flight_id"].astype("string").str.strip().str.upper()

flight_id_pattern = r"^[A-Z0-9]{5}$"

flights["flight_id_valid"] = flights["flight_id"].str.match(
    flight_id_pattern,
    na=False
)

print("Valid Flight IDs:", flights["flight_id_valid"].sum())
print("Invalid Flight IDs:", (~flights["flight_id_valid"]).sum())

display(
    flights.loc[~flights["flight_id_valid"], ["flight_id", "airline"]]
    .head(20)
)

Valid Flight IDs: 1020
Invalid Flight IDs: 0


,flight_id,airline


In [62]:
#Standardize text columns
text_columns = [
    "airline",
    "source",
    "destination"
]

for column in text_columns:
    flights[column] = (
        flights[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

flights["airline"] = flights["airline"].fillna("UNKNOWN")

print("Text standardization completed")

Text standardization completed


In [63]:
#Convert timestamps
flights["departure_time"] = pd.to_datetime(
    flights["departure_time"],
    errors="coerce"
)

flights["arrival_time"] = pd.to_datetime(
    flights["arrival_time"],
    errors="coerce"
)

print(flights[[
    "departure_time",
    "arrival_time"
]].dtypes)

departure_time    datetime64[ns]
arrival_time      datetime64[ns]
dtype: object


In [64]:
#Duration calculation
flights["duration_calculated"] = (
    flights["arrival_time"] - flights["departure_time"]
)

flights["duration_minutes"] = (
    flights["duration_calculated"]
    .dt.total_seconds() / 60
)

flights["duration_hours"] = (
    flights["duration_minutes"] / 60
)

display(
    flights[
        [
            "flight_id",
            "departure_time",
            "arrival_time",
            "duration_minutes",
            "duration_hours"
        ]
    ].head(10)
)

,flight_id,departure_time,arrival_time,duration_minutes,duration_hours
0,SJ010,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,174.0,2.900000
1,AI155,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,108.0,1.800000
2,UK094,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,105.0,1.750000
3,AI245,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,156.0,2.600000
4,AI192,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,299.0,4.983333
5,SJ158,2026-04-20 23:05:41.703,2026-04-21 01:24:41.703,139.0,2.316667
6,6F196,2026-04-20 23:04:41.703,2026-04-21 00:47:41.703,103.0,1.716667
7,AI080,2026-04-20 23:03:41.702,2026-04-21 00:35:41.702,92.0,1.533333
8,6F025,2026-04-20 23:02:41.701,2026-04-20 23:57:41.701,55.0,0.916667
9,6F251,2026-04-20 22:56:41.703,2026-04-20 23:37:41.703,41.0,0.683333


In [65]:
#Overnight flight detection
flights["is_overnight"] = (
    flights["arrival_time"].dt.date >
    flights["departure_time"].dt.date
)

print(
    "Overnight flights:",
    flights["is_overnight"].sum()
)

print(
    "Same-day flights:",
    (~flights["is_overnight"]).sum()
)

Overnight flights: 124
Same-day flights: 896


In [66]:
#Check negative durations
negative_duration = flights[
    flights["duration_minutes"] < 0
]

print(
    "Negative duration records:",
    len(negative_duration)
)

display(negative_duration.head())

Negative duration records: 1


,flight_id,airline,source,destination,departure_time,arrival_time,duration,flight_id_valid,duration_calculated,duration_minutes,duration_hours,is_overnight
355,SJ192,SPICEJET,HYD,BOM,2026-04-19 18:45:42,2026-04-18 23:45:42,1899-12-29 05:00:00,True,-1 days +05:00:00,-1140.0,-19.0,False


In [20]:

print(flights["duration"].head(20))
print(flights["duration"].dtype)

0     02:54:00
1     01:48:00
2     01:45:00
3     02:36:00
4     04:59:00
5     02:19:00
6     01:43:00
7     01:32:00
8     00:55:00
9     00:41:00
10    02:28:00
11    04:27:00
12    03:01:00
13    02:21:00
14    03:46:00
15    01:38:00
16    03:43:00
17    01:46:00
18    00:35:00
19    03:52:00
Name: duration, dtype: object
object


In [67]:
#Compare original vs calculated duration
flights["duration_original_minutes"] = pd.to_timedelta(
    flights["duration"].astype(str),
    errors="coerce"
).dt.total_seconds() / 60

flights["duration_difference"] = (
    flights["duration_minutes"] -
    flights["duration_original_minutes"]
)

display(
    flights[
        [
            "flight_id",
            "duration",
            "duration_original_minutes",
            "duration_minutes",
            "duration_difference"
        ]
    ].head(20)
)

,flight_id,duration,duration_original_minutes,duration_minutes,duration_difference
0,SJ010,02:54:00,174.0,174.0,0.0
1,AI155,01:48:00,108.0,108.0,0.0
2,UK094,01:45:00,105.0,105.0,0.0
3,AI245,02:36:00,156.0,156.0,0.0
4,AI192,04:59:00,299.0,299.0,0.0
5,SJ158,02:19:00,139.0,139.0,0.0
6,6F196,01:43:00,103.0,103.0,0.0
7,AI080,01:32:00,92.0,92.0,0.0
8,6F025,00:55:00,55.0,55.0,0.0
9,6F251,00:41:00,41.0,41.0,0.0


In [68]:
#Create route
flights["route"] = (
    flights["source"] +
    " → " +
    flights["destination"]
)

display(
    flights[
        [
            "flight_id",
            "source",
            "destination",
            "route"
        ]
    ].head()
)

,flight_id,source,destination,route
0,SJ010,CCU,MAA,CCU → MAA
1,AI155,BOM,CCU,BOM → CCU
2,UK094,BOM,CCU,BOM → CCU
3,AI245,BOM,CCU,BOM → CCU
4,AI192,MAA,BOM,MAA → BOM


In [69]:
#Create analytical date fields
flights["departure_date"] = flights["departure_time"].dt.date
flights["arrival_date"] = flights["arrival_time"].dt.date

flights["departure_hour"] = (
    flights["departure_time"].dt.hour
)

flights["arrival_hour"] = (
    flights["arrival_time"].dt.hour
)

flights["departure_day"] = (
    flights["departure_time"].dt.day_name()
)

display(
    flights[
        [
            "flight_id",
            "departure_date",
            "departure_hour",
            "departure_day",
            "is_overnight"
        ]
    ].head()
)

,flight_id,departure_date,departure_hour,departure_day,is_overnight
0,SJ010,2026-04-20,23,Monday,True
1,AI155,2026-04-20,23,Monday,True
2,UK094,2026-04-20,23,Monday,True
3,AI245,2026-04-20,23,Monday,True
4,AI192,2026-04-20,23,Monday,True


In [70]:
#PII analysis

passengers_clean = passengers.copy()
bookings_clean = bookings.copy()

In [71]:
#Create masked passenger dataset
def hash_value(value):
    if pd.isna(value):
        return None

    return hashlib.sha256(
        str(value).encode("utf-8")
    ).hexdigest()

In [72]:
passengers_clean["passenger_key"] = (
    passengers_clean["passenger_id"]
    .apply(hash_value)
)

In [73]:
#Mask Aadhaar
def mask_aadhaar(value):
    if pd.isna(value):
        return None

    value = str(value)
    return "*" * max(0, len(value) - 4) + value[-4:]

passengers_clean["aadhaar_masked"] = (
    passengers_clean["aadhaar_id"]
    .apply(mask_aadhaar)
)

In [74]:
#Remove unnecessary PII
passenger_analytics = passengers_clean[
    [
        "passenger_id",
        "passenger_key",
        "age",
        "gender"
    ]
].copy()

display(passenger_analytics.head())

,passenger_id,passenger_key,age,gender
0,P1000,031a60e86398d41d18e79ffcedc168a3e137eec8e808ff...,52,F
1,P1001,61d0f276b7160f04f92ce7d852f04296612a23d85e1041...,15,M
2,P1002,c5baeafae225ad4d584ec322dd47d1bdde1c4e989d7169...,72,M
3,P1003,eeae6e0fb3e9e763d79dde942b8480b918448aefc3d538...,61,F
4,P1004,6533dc26f5dafb4abea1d39bc6d918a37c172db64bb11d...,21,M


In [75]:
#Booking PII protection
booking_analytics = bookings_clean[
    [
        "booking_id",
        "passenger_id",
        "flight_id",
        "booking_date",
        "status",
        "seat_number"
    ]
].copy()

display(booking_analytics.head())

,booking_id,passenger_id,flight_id,booking_date,status,seat_number
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,3D
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,18A
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,30C
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,33A
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,25C


In [77]:
#Clean booking status
booking_analytics["status"] = (
    booking_analytics["status"]
    .astype("string")
    .str.strip()
    .str.upper()
)

booking_analytics["status"] = (
    booking_analytics["status"]
    .fillna("UNKNOWN")
)

print(
    booking_analytics["status"].value_counts()
)

status
CONFIRMED    320
CANCELLED    314
PENDING      291
UNKNOWN       45
INVALID       30
Name: count, dtype: Int64


In [78]:
#Payment cleaning
payments_clean = payments.copy()

payments_clean["payment_method"] = (
    payments_clean["payment_method"]
    .astype("string")
    .str.strip()
    .str.upper()
)

payments_clean["amount"] = pd.to_numeric(
    payments_clean["amount"],
    errors="coerce"
)

payments_clean["amount"] = (
    payments_clean["amount"].fillna(0)
)

display(payments_clean.head())

,payment_id,booking_id,amount,payment_method
0,PAY1000,B1116,9883.49,NETBANKING
1,PAY1001,B1738,8457.96,NETBANKING
2,PAY1002,B1873,6495.37,UPI
3,PAY1003,B1914,5079.38,NETBANKING
4,PAY1004,B1967,12518.31,CARD


In [79]:
#Referential integrity
booking_flight_check = booking_analytics[
    ~booking_analytics["flight_id"].isin(
        flights["flight_id"]
    )
]

print(
    "Bookings with invalid flight IDs:",
    len(booking_flight_check)
)

display(booking_flight_check.head())

Bookings with invalid flight IDs: 0


,booking_id,passenger_id,flight_id,booking_date,status,seat_number


In [80]:
#Booking → Passenger
booking_passenger_check = booking_analytics[
    ~booking_analytics["passenger_id"].isin(
        passengers["passenger_id"]
    )
]

print(
    "Bookings with invalid passenger IDs:",
    len(booking_passenger_check)
)

display(booking_passenger_check.head())

Bookings with invalid passenger IDs: 0


,booking_id,passenger_id,flight_id,booking_date,status,seat_number


In [81]:
#Payment → Booking
payment_booking_check = payments_clean[
    ~payments_clean["booking_id"].isin(
        booking_analytics["booking_id"]
    )
]

print(
    "Payments with invalid booking IDs:",
    len(payment_booking_check)
)

display(payment_booking_check.head())

Payments with invalid booking IDs: 0


,payment_id,booking_id,amount,payment_method


In [82]:
#Create data-quality flags
flights["data_quality_flag"] = "VALID"

flights.loc[
    flights["flight_id_valid"] == False,
    "data_quality_flag"
] = "INVALID_FLIGHT_ID"

flights.loc[
    flights["departure_time"].isna() |
    flights["arrival_time"].isna(),
    "data_quality_flag"
] = "INVALID_TIMESTAMP"

flights.loc[
    flights["duration_minutes"] < 0,
    "data_quality_flag"
] = "NEGATIVE_DURATION"

flights.loc[
    flights["duration_minutes"].isna(),
    "data_quality_flag"
] = "MISSING_DURATION"

print(
    flights["data_quality_flag"].value_counts()
)

data_quality_flag
VALID                1019
NEGATIVE_DURATION       1
Name: count, dtype: int64


In [83]:
#Final analytical flight table
fact_flights = flights[
    [
        "flight_id",
        "airline",
        "source",
        "destination",
        "route",
        "departure_time",
        "arrival_time",
        "departure_date",
        "arrival_date",
        "departure_hour",
        "arrival_hour",
        "departure_day",
        "duration_minutes",
        "duration_hours",
        "is_overnight",
        "data_quality_flag"
    ]
].copy()

display(fact_flights.head())

,flight_id,airline,source,destination,route,departure_time,arrival_time,departure_date,arrival_date,departure_hour,arrival_hour,departure_day,duration_minutes,duration_hours,is_overnight,data_quality_flag
0,SJ010,SPICEJET,CCU,MAA,CCU → MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,2026-04-20,2026-04-21,23,2,Monday,174.0,2.900000,True,VALID
1,AI155,AIR INDIA,BOM,CCU,BOM → CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,2026-04-20,2026-04-21,23,1,Monday,108.0,1.800000,True,VALID
2,UK094,VISTARA,BOM,CCU,BOM → CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,2026-04-20,2026-04-21,23,1,Monday,105.0,1.750000,True,VALID
3,AI245,AIR INDIA,BOM,CCU,BOM → CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,2026-04-20,2026-04-21,23,1,Monday,156.0,2.600000,True,VALID
4,AI192,AIR INDIA,MAA,BOM,MAA → BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,2026-04-20,2026-04-21,23,4,Monday,299.0,4.983333,True,VALID


In [84]:
#Create airline dimension
dim_airline = (
    fact_flights[
        ["airline"]
    ]
    .drop_duplicates()
    .sort_values("airline")
    .reset_index(drop=True)
)

dim_airline["airline_key"] = (
    dim_airline.index + 1
)

display(dim_airline)

,airline,airline_key
0,AIR INDIA,1
1,INDIGO,2
2,SPICEJET,3
3,UNKNOWN,4
4,VISTARA,5


In [85]:
#Create route dimension
dim_route = (
    fact_flights[
        [
            "source",
            "destination",
            "route"
        ]
    ]
    .drop_duplicates()
    .sort_values("route")
    .reset_index(drop=True)
)

dim_route["route_key"] = (
    dim_route.index + 1
)

display(dim_route.head(20))

,source,destination,route,route_key
0,BLR,BOM,BLR → BOM,1
1,BLR,CCU,BLR → CCU,2
2,BLR,DEL,BLR → DEL,3
3,BLR,HYD,BLR → HYD,4
4,BLR,MAA,BLR → MAA,5
5,BOM,BLR,BOM → BLR,6
6,BOM,CCU,BOM → CCU,7
7,BOM,DEL,BOM → DEL,8
8,BOM,HYD,BOM → HYD,9
9,BOM,MAA,BOM → MAA,10


In [86]:
#KPI preparation
total_flights = len(fact_flights)

total_routes = fact_flights["route"].nunique()

total_airlines = fact_flights["airline"].nunique()

average_duration = (
    fact_flights["duration_minutes"].mean()
)

overnight_flights = (
    fact_flights["is_overnight"].sum()
)

anomalies = (
    fact_flights["data_quality_flag"] != "VALID"
).sum()

print("Total Flights:", total_flights)
print("Total Routes:", total_routes)
print("Total Airlines:", total_airlines)
print("Average Duration:", round(average_duration, 2), "minutes")
print("Overnight Flights:", overnight_flights)
print("Anomalies:", anomalies)

Total Flights: 1020
Total Routes: 30
Total Airlines: 5
Average Duration: 162.92 minutes
Overnight Flights: 124
Anomalies: 1


In [87]:
#Route traffic
route_traffic = (
    fact_flights
    .groupby("route")
    .size()
    .reset_index(name="flight_count")
    .sort_values(
        "flight_count",
        ascending=False
    )
)

display(route_traffic.head(20))

,route,flight_count
6,BOM → CCU,90
12,CCU → DEL,74
25,MAA → BLR,65
0,BLR → BOM,62
24,HYD → MAA,57
18,DEL → HYD,55
23,HYD → DEL,42
7,BOM → DEL,39
11,CCU → BOM,34
16,DEL → BOM,29


In [88]:
#Airline distribution
airline_distribution = (
    fact_flights
    .groupby("airline")
    .size()
    .reset_index(name="flight_count")
    .sort_values(
        "flight_count",
        ascending=False
    )
)

display(airline_distribution)

,airline,flight_count
1,INDIGO,249
2,SPICEJET,240
0,AIR INDIA,236
4,VISTARA,223
3,UNKNOWN,72


In [89]:
#Average duration by airline
duration_by_airline = (
    fact_flights
    .groupby("airline")["duration_minutes"]
    .mean()
    .reset_index()
    .sort_values(
        "duration_minutes",
        ascending=False
    )
)

display(duration_by_airline)

,airline,duration_minutes
1,INDIGO,167.702951
3,UNKNOWN,165.083333
0,AIR INDIA,163.334872
4,VISTARA,162.233362
2,SPICEJET,157.537624


In [90]:
#Anomaly summary
anomaly_summary = (
    fact_flights[
        fact_flights["data_quality_flag"] != "VALID"
    ]
    .groupby("data_quality_flag")
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
)

display(anomaly_summary)

,data_quality_flag,count
0,NEGATIVE_DURATION,1


In [91]:
#Final validation
print("FINAL DATA VALIDATION")
print("=" * 50)

print(
    "Duplicate flight IDs:",
    fact_flights["flight_id"].duplicated().sum()
)

print(
    "Missing flight IDs:",
    fact_flights["flight_id"].isna().sum()
)

print(
    "Missing departure timestamps:",
    fact_flights["departure_time"].isna().sum()
)

print(
    "Missing arrival timestamps:",
    fact_flights["arrival_time"].isna().sum()
)

print(
    "Negative durations:",
    (fact_flights["duration_minutes"] < 0).sum()
)

print(
    "Overnight flights:",
    fact_flights["is_overnight"].sum()
)

print(
    "Anomalous records:",
    (
        fact_flights["data_quality_flag"] != "VALID"
    ).sum()
)

FINAL DATA VALIDATION
Duplicate flight IDs: 16
Missing flight IDs: 0
Missing departure timestamps: 0
Missing arrival timestamps: 0
Negative durations: 1
Overnight flights: 124
Anomalous records: 1


In [46]:
os.makedirs("output", exist_ok=True)

In [47]:
fact_flights.to_csv(
    "output/fact_flights.csv",
    index=False
)

booking_analytics.to_csv(
    "output/fact_bookings.csv",
    index=False
)

payments_clean.to_csv(
    "output/fact_payments.csv",
    index=False
)

passenger_analytics.to_csv(
    "output/dim_passenger.csv",
    index=False
)

dim_airline.to_csv(
    "output/dim_airline.csv",
    index=False
)

dim_route.to_csv(
    "output/dim_route.csv",
    index=False
)

print("All analytical datasets exported successfully")

All analytical datasets exported successfully


In [92]:
#Create a KPI file
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Flights",
        "Total Routes",
        "Total Airlines",
        "Average Flight Duration (Minutes)",
        "Overnight Flights",
        "Data Quality Anomalies"
    ],
    "Value": [
        total_flights,
        total_routes,
        total_airlines,
        round(average_duration, 2),
        overnight_flights,
        anomalies
    ]
})

display(kpi_summary)

kpi_summary.to_csv(
    "output/kpi_summary.csv",
    index=False
)

,KPI,Value
0,Total Flights,1020.00
1,Total Routes,30.00
2,Total Airlines,5.00
3,Average Flight Duration (Minutes),162.92
4,Overnight Flights,124.00
5,Data Quality Anomalies,1.00


In [49]:
from google.colab import files

files.download("output/fact_flights.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [50]:
!zip -r ASG_Airlines_Analytics_Output.zip output

  adding: output/ (stored 0%)
  adding: output/kpi_summary.csv (deflated 23%)
  adding: output/dim_route.csv (deflated 64%)
  adding: output/fact_bookings.csv (deflated 76%)
  adding: output/fact_flights.csv (deflated 86%)
  adding: output/dim_airline.csv (deflated 7%)
  adding: output/fact_payments.csv (deflated 66%)
  adding: output/dim_passenger.csv (deflated 46%)


In [51]:
files.download(
    "ASG_Airlines_Analytics_Output.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>